# Diffusion-Based Novel View Synthesis — Zero123++

Generates 4 novel views per input image using Zero123++.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload a zip of your `diffusion_nvs/inputs/` folder when prompted in Cell 2

**Output:** download `nvs_outputs.zip` from the last cell — drop it into your local `diffusion_nvs/outputs/`.

In [ ]:
# Cell 1 — Fix environment & install dependencies
#
# Run this cell. If it prints "Imports OK", continue to Cell 2.
# If it kills the session, just re-run this cell once more after the restart.

import torch, subprocess, sys

# Step 1: Fix torchvision CUDA mismatch (Colab ships mismatched versions)
cuda_tag = "cu" + torch.version.cuda.replace(".", "")
print(f"PyTorch CUDA: {torch.version.cuda} — reinstalling torchvision for {cuda_tag} ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "torchvision", "--index-url",
     f"https://download.pytorch.org/whl/{cuda_tag}"],
    check=False
)

# Step 2: Install latest diffusers stack — no version pins, let pip resolve
print("Installing diffusers stack ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "diffusers", "transformers", "accelerate", "huggingface_hub",
     "einops", "Pillow", "tqdm"],
    check=True
)

# Step 3: Verify imports (if this fails the session restarts automatically)
try:
    import torchvision
    import diffusers, transformers
    from diffusers import DiffusionPipeline
    print(f"\ndiffusers   : {diffusers.__version__}")
    print(f"transformers: {transformers.__version__}")
    print(f"PyTorch     : {torch.__version__}  CUDA: {torch.version.cuda}")
    if torch.cuda.is_available():
        print(f"GPU         : {torch.cuda.get_device_name(0)}")
        print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("WARNING: No GPU — Runtime -> Change runtime type -> T4 GPU")
    print("\nImports OK — continue to Cell 2")
except Exception as e:
    print(f"\nImport error: {e}")
    print("Restarting session so new packages take effect ...")
    import os; os.kill(os.getpid(), 9)

In [ ]:
# Cell 2 — Upload your inputs zip
#
# On your Mac, create the zip from the project root:
#   zip -rj inputs.zip diffusion_nvs/inputs/
# Then upload inputs.zip when the file picker appears below.

from google.colab import files
import zipfile
from pathlib import Path

INPUT_DIR  = Path('/content/inputs')
OUTPUT_DIR = Path('/content/outputs/novel_views')
GRID_DIR   = Path('/content/outputs/grids')

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GRID_DIR.mkdir(parents=True, exist_ok=True)

print('Upload your inputs.zip now...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    for member in z.namelist():
        fname = Path(member).name
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')) and fname:
            with z.open(member) as src, open(INPUT_DIR / fname, 'wb') as dst:
                dst.write(src.read())

images = sorted(INPUT_DIR.glob('*.jpg')) + sorted(INPUT_DIR.glob('*.png'))
print(f'Loaded {len(images)} input images')
assert len(images) > 0, 'No images found — check your zip contains input_*.jpg/png files'

In [ ]:
# Cell 3 — Configuration
N_VIEWS        = 4     # novel views to keep per image (max 6)
GUIDANCE_SCALE = 4.0   # classifier-free guidance
N_STEPS        = 75    # denoising steps (28 = fast, 75 = high quality)
VIEW_SIZE      = 320   # px per view — T4 16GB handles 320 fine

In [ ]:
# Cell 4 — Load Zero123++ pipeline
from diffusers import DiffusionPipeline, EulerAncestralDiscreteScheduler

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if device == 'cuda' else torch.float32

print(f'Loading Zero123++ on {device} ({dtype})...')
pipe = DiffusionPipeline.from_pretrained(
    'sudo-ai/zero123plus-v1.2',
    custom_pipeline='sudo-ai/zero123plus-pipeline',
    torch_dtype=dtype,
    trust_remote_code=True,
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
    pipe.scheduler.config,
    timestep_spacing='trailing',
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()
print('Pipeline ready.')

In [ ]:
# Cell 5 — Helper functions
from PIL import Image
import numpy as np

AZIMUTHS  = [30, 90, 150, 210, 270, 330]   # baked into Zero123++ weights
ELEVATION = 20                              # approximate, fixed by model


def preprocess_image(path, size=320):
    img = Image.open(path).convert('RGBA')
    w, h = img.size
    crop = min(w, h)
    img = img.crop(((w-crop)//2, (h-crop)//2, (w+crop)//2, (h+crop)//2))
    bg = Image.new('RGBA', img.size, (255, 255, 255, 255))
    bg.paste(img, mask=img.split()[3])
    return bg.convert('RGB').resize((size, size), Image.LANCZOS)


def split_grid(grid_img, n_views):
    """Split 2-col x 3-row Zero123++ output grid into individual views."""
    w, h = grid_img.size
    cell_w, cell_h = w // 2, h // 3
    views = []
    for row in range(3):
        for col in range(2):
            box = (col*cell_w, row*cell_h, (col+1)*cell_w, (row+1)*cell_h)
            views.append(grid_img.crop(box))
    return views[:n_views]


@torch.no_grad()
def generate_views(pipe, input_image, n_views, guidance_scale, n_steps, view_size):
    result = pipe(
        input_image,
        num_inference_steps=n_steps,
        guidance_scale=guidance_scale,
        width=view_size * 2,
        height=view_size * 3,
    )
    return split_grid(result.images[0], n_views)


print('Helpers defined.')

In [ ]:
# Cell 6 — Generate novel views
from tqdm.notebook import tqdm

selected_azimuths = AZIMUTHS[:N_VIEWS]
print(f'Azimuths : {selected_azimuths} deg  |  Elevation: +{ELEVATION} deg  |  View size: {VIEW_SIZE}px')
print(f'Total    : {len(images)} images x {N_VIEWS} views = {len(images)*N_VIEWS} outputs')
print()

total_generated = 0
for img_path in tqdm(images, desc='Images'):
    stem = img_path.stem
    input_img = preprocess_image(img_path, size=320)
    input_img.save(OUTPUT_DIR / f'{stem}_input.png')

    views = generate_views(pipe, input_img, N_VIEWS, GUIDANCE_SCALE, N_STEPS, VIEW_SIZE)

    for view, azim in zip(views, selected_azimuths):
        view.save(OUTPUT_DIR / f'{stem}_elev+{ELEVATION}_azim+{azim:03d}.png')
        total_generated += 1

print(f'\nGenerated {total_generated} novel views -> {OUTPUT_DIR}')

In [ ]:
# Cell 7 — Create comparison grids
from PIL import ImageDraw

def make_grid(input_img, novel_views, labels, cell_size=256):
    n = 1 + len(novel_views)
    label_h = 36
    grid = Image.new('RGB', (cell_size * n, cell_size + label_h), (255, 255, 255))
    draw = ImageDraw.Draw(grid)
    for i, (img, label) in enumerate(zip([input_img] + novel_views, ['Input'] + labels)):
        grid.paste(img.resize((cell_size, cell_size), Image.LANCZOS), (i * cell_size, label_h))
        draw.text((i * cell_size + cell_size // 2, 10), label, fill=(0, 0, 0), anchor='mm')
    return grid

input_images = sorted(OUTPUT_DIR.glob('*_input.png'))
for input_path in tqdm(input_images, desc='Grids'):
    stem = input_path.stem.replace('_input', '')
    novel_paths = sorted(OUTPUT_DIR.glob(f'{stem}_elev*.png'))
    if not novel_paths:
        continue
    input_img  = Image.open(input_path).convert('RGB')
    novel_imgs = [Image.open(p).convert('RGB') for p in novel_paths]
    labels = []
    for p in novel_paths:
        parts = p.stem.split('_')
        elev = next((x for x in parts if x.startswith('elev')), '')
        azim = next((x for x in parts if x.startswith('azim')), '')
        labels.append(f'{elev} {azim}')
    grid = make_grid(input_img, novel_imgs, labels)
    grid.save(GRID_DIR / f'{stem}_grid.png')

print(f'{len(input_images)} grids saved -> {GRID_DIR}')

In [ ]:
# Cell 8 — Preview sample grids
import matplotlib.pyplot as plt

grids = sorted(GRID_DIR.glob('*_grid.png'))
samples = [grids[0], grids[len(grids)//2], grids[-1]] if len(grids) >= 3 else grids
fig, axes = plt.subplots(len(samples), 1, figsize=(20, 5 * len(samples)))
if len(samples) == 1:
    axes = [axes]
for ax, p in zip(axes, samples):
    ax.imshow(Image.open(p))
    ax.set_title(p.name)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 — Download all outputs as a zip
# On your Mac: unzip nvs_outputs.zip -d diffusion_nvs/outputs/
import shutil

zip_out = '/content/nvs_outputs'
shutil.make_archive(zip_out, 'zip', '/content/outputs')
files.download(f'{zip_out}.zip')
print('Download started — check your browser downloads.')